# 13 — Gold Features: Camada de Feature Engineering

**Credit Risk Intelligence Platform** — Camada Gold

Este notebook constrói a camada Gold do projeto Credit Risk, consolidando informações de todas as tabelas Silver em features agregadas por `SK_ID_CURR`, prontas para Machine Learning.

## Pipeline

```
credit_risk.silver.* (8 tabelas)  →  credit_risk.gold.credit_risk_features_train
                                      credit_risk.gold.credit_risk_features_test
```

## Princípios

> **1 SK_ID_CURR = 1 linha na Gold** — sem row explosion.
> Cada tabela histórica é agregada individualmente por `SK_ID_CURR` antes do JOIN final.
> **TARGET não é utilizado como feature** — validação explícita contra data leakage.
> Bronze e Silver **NÃO são modificadas**.
> Nenhuma feature de ML é criada artificialmente — apenas agregações e razões justificáveis.
> NULLs são preservados quando têm significado diferente de zero.

In [0]:
# ============================================================================
# CÉLULA 1 — Configuração e Imports
# ============================================================================
from pyspark.sql import functions as F, types as T, Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, LongType, DateType, BooleanType)
from datetime import datetime, timezone
import uuid

# ----------------------------------------------------------------------------
# Identificadores de execução
# ----------------------------------------------------------------------------
PIPELINE_VERSION = "gold_v1.0"
NOTEBOOK_NAME = "13_gold_features"
EXECUTION_ID = str(uuid.uuid4())
BATCH_ID = f"gold_features_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
EXECUTION_TIMESTAMP = datetime.now(timezone.utc)
EXEC_START = EXECUTION_TIMESTAMP

# ----------------------------------------------------------------------------
# Tabelas de origem (Silver)
# ----------------------------------------------------------------------------
SILVER_APP_TRAIN = "credit_risk.silver.application_train"
SILVER_APP_TEST = "credit_risk.silver.application_test"
SILVER_BUREAU = "credit_risk.silver.bureau"
SILVER_BUREAU_BALANCE = "credit_risk.silver.bureau_balance"
SILVER_PREV_APP = "credit_risk.silver.previous_application"
SILVER_POS_CASH = "credit_risk.silver.pos_cash_balance"
SILVER_CC_BALANCE = "credit_risk.silver.credit_card_balance"
SILVER_INST_PAY = "credit_risk.silver.installments_payments"

ALL_SILVER_TABLES = [
    SILVER_APP_TRAIN, SILVER_APP_TEST, SILVER_BUREAU, SILVER_BUREAU_BALANCE,
    SILVER_PREV_APP, SILVER_POS_CASH, SILVER_CC_BALANCE, SILVER_INST_PAY
]

# ----------------------------------------------------------------------------
# Tabelas de destino (Gold)
# ----------------------------------------------------------------------------
GOLD_SCHEMA = "credit_risk.gold"
GOLD_TRAIN_TABLE = f"{GOLD_SCHEMA}.credit_risk_features_train"
GOLD_TEST_TABLE = f"{GOLD_SCHEMA}.credit_risk_features_test"
GOLD_FEATURE_CATALOG = f"{GOLD_SCHEMA}.feature_catalog"
GOLD_AUDIT_TABLE = f"{GOLD_SCHEMA}.audit_features"

# ----------------------------------------------------------------------------
# Colunas de controle Silver a remover
# ----------------------------------------------------------------------------
SILVER_CONTROL_COLS = [
    "silver_processing_timestamp", "silver_processing_date",
    "silver_pipeline_version", "source_table", "record_hash"
]

# ----------------------------------------------------------------------------
# Acumulador de features para catálogo
# ----------------------------------------------------------------------------
FEATURE_CATALOG = []

def register_feature(name, source_table, source_columns, definition, agg_method,
                      data_type, business_meaning, null_semantics,
                      leakage_check="PASS", ml_feature_flag=True):
    """Registra uma feature no catálogo."""
    FEATURE_CATALOG.append({
        "feature_name": name,
        "source_table": source_table,
        "source_columns": source_columns,
        "feature_definition": definition,
        "aggregation_method": agg_method,
        "data_type": data_type,
        "business_meaning": business_meaning,
        "null_semantics": null_semantics,
        "leakage_check": leakage_check,
        "ml_feature_flag": ml_feature_flag,
        "created_at": EXECUTION_TIMESTAMP,
    })

print(f"⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline: {PIPELINE_VERSION}")
print(f"📊 Tabelas Silver de origem: {len(ALL_SILVER_TABLES)}")
print("✅ Configuração inicial concluída!")

In [0]:
# ============================================================================
# CÉLULA 2 — Criação/Verificação do Schema Gold
# ============================================================================
# Cria o schema credit_risk.gold se não existir.

print("=" * 70)
print("CRIAÇÃO DO SCHEMA GOLD")
print("=" * 70)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
print(f"✅ Schema {GOLD_SCHEMA} verificado/criado.")

# Listar tabelas existentes no schema Gold
print(f"\nTabelas existentes em {GOLD_SCHEMA}:")
gold_tables = spark.sql(f"SHOW TABLES IN {GOLD_SCHEMA}").collect()
if gold_tables:
    for t in gold_tables:
        print(f"   {t.database}.{t.tableName}")
else:
    print("   (nenhuma tabela ainda)")

print("\n✅ Schema Gold pronto!")

In [0]:
# ============================================================================
# CÉLULA 3 — Inspeção dos Schemas Silver
# ============================================================================
# Inspeciona os schemas reais das tabelas Silver para confirmar colunas disponíveis.

sep = "─" * 70
print("=" * 70)
print("INSPEÇÃO DOS SCHEMAS SILVER")
print("=" * 70)

# Armazenar colunas disponíveis por tabela
SILVER_COLUMNS = {}

for tname in ALL_SILVER_TABLES:
    df = spark.table(tname)
    cols = [f.name for f in df.schema.fields]
    SILVER_COLUMNS[tname] = cols
    rc = df.count()
    print(f"\n   {tname}: {rc:,} rows, {len(cols)} cols")
    # Mostrar colunas de dados (excluindo controle Silver)
    data_cols = [c for c in cols if c not in SILVER_CONTROL_COLS]
    print(f"   Colunas de dados: {len(data_cols)}")

# Resumo das colunas-chave para feature engineering
print(f"\n{sep}")
print("COLUNAS-CHAVE PARA FEATURE ENGINEERING")
print(sep)

key_checks = {
    SILVER_APP_TRAIN: ["SK_ID_CURR", "TARGET", "DAYS_BIRTH", "DAYS_EMPLOYED",
                       "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
                       "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
                       "CNT_CHILDREN", "CNT_FAM_MEMBERS"],
    SILVER_BUREAU: ["SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_ACTIVE",
                    "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_MAX_OVERDUE",
                    "AMT_CREDIT_SUM_OVERDUE", "CREDIT_DAY_OVERDUE", "DAYS_CREDIT",
                    "CNT_CREDIT_PROLONG"],
    SILVER_BUREAU_BALANCE: ["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"],
    SILVER_PREV_APP: ["SK_ID_CURR", "SK_ID_PREV", "NAME_CONTRACT_STATUS",
                      "AMT_APPLICATION", "AMT_CREDIT", "AMT_ANNUITY",
                      "AMT_DOWN_PAYMENT", "DAYS_DECISION", "CNT_PAYMENT"],
    SILVER_POS_CASH: ["SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE",
                      "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE",
                      "NAME_CONTRACT_STATUS", "SK_DPD", "SK_DPD_DEF"],
    SILVER_CC_BALANCE: ["SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE",
                        "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
                        "AMT_DRAWINGS_CURRENT", "AMT_PAYMENT_CURRENT",
                        "AMT_TOTAL_RECEIVABLE", "NAME_CONTRACT_STATUS",
                        "SK_DPD", "SK_DPD_DEF"],
    SILVER_INST_PAY: ["SK_ID_CURR", "SK_ID_PREV", "DAYS_INSTALMENT",
                      "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT",
                      "FLAG_PAYMENT_MISSING"],
}

for tname, keys in key_checks.items():
    available = SILVER_COLUMNS.get(tname, [])
    existing = [k for k in keys if k in available]
    missing = [k for k in keys if k not in available]
    short_name = tname.split(".")[-1]
    print(f"\n   {short_name}:")
    print(f"      Existentes: {len(existing)}/{len(keys)}")
    if missing:
        print(f"      Ausentes: {missing}")

print("\n✅ Inspeção de schemas concluída!")

In [0]:
# ============================================================================
# CÉLULA 4 — Preparação das Features da Aplicação
# ============================================================================
# Seleciona colunas relevantes de application_train/test e cria features derivadas.
# Não utiliza TARGET como feature. Remove colunas de controle Silver.

def prepare_application_features(app_df, table_name):
    """Prepara features da aplicação: seleção + features derivadas."""
    cols = SILVER_COLUMNS[table_name]
    
    # Selecionar colunas de dados relevantes (excluir controle Silver e TARGET)
    feature_cols = [c for c in cols if c not in SILVER_CONTROL_COLS and c != "TARGET"]
    df = app_df.select(*feature_cols)
    
    # Features derivadas (com proteção contra divisão por zero)
    # age_years a partir de DAYS_BIRTH (negativo → positivo)
    if "DAYS_BIRTH" in cols:
        df = df.withColumn("age_years", F.round(F.col("DAYS_BIRTH") * -1.0 / 365.25, 2))
        register_feature("age_years", table_name, "DAYS_BIRTH",
            "Idade em anos (-DAYS_BIRTH / 365.25)", "derivation",
            "double", "Idade do cliente", "Sempre preenchido (DAYS_BIRTH sem NULL)")
    
    # employed_years a partir de DAYS_EMPLOYED (tratar anomalia 365243)
    if "DAYS_EMPLOYED" in cols:
        df = df.withColumn("employed_years",
            F.when(F.col("DAYS_EMPLOYED") == 365243, None)
             .otherwise(F.round(F.col("DAYS_EMPLOYED") * -1.0 / 365.25, 2)))
        register_feature("employed_years", table_name, "DAYS_EMPLOYED",
            "Anos empregado (-DAYS_EMPLOYED / 365.25, 365243 → NULL)", "derivation",
            "double", "Tempo de emprego", "NULL quando DAYS_EMPLOYED = 365243 (aposentado)")
    
    # credit_income_ratio = AMT_CREDIT / AMT_INCOME_TOTAL
    if "AMT_CREDIT" in cols and "AMT_INCOME_TOTAL" in cols:
        df = df.withColumn("credit_income_ratio",
            F.when(F.col("AMT_INCOME_TOTAL").isNull() | (F.col("AMT_INCOME_TOTAL") == 0), None)
             .otherwise(F.round(F.col("AMT_CREDIT") / F.col("AMT_INCOME_TOTAL"), 4)))
        register_feature("credit_income_ratio", table_name, "AMT_CREDIT, AMT_INCOME_TOTAL",
            "AMT_CREDIT / AMT_INCOME_TOTAL", "derivation",
            "double", "Razão crédito/renda", "NULL quando renda é NULL ou zero")
    
    # annuity_income_ratio = AMT_ANNUITY / AMT_INCOME_TOTAL
    if "AMT_ANNUITY" in cols and "AMT_INCOME_TOTAL" in cols:
        df = df.withColumn("annuity_income_ratio",
            F.when(F.col("AMT_INCOME_TOTAL").isNull() | (F.col("AMT_INCOME_TOTAL") == 0), None)
             .otherwise(F.round(F.col("AMT_ANNUITY") / F.col("AMT_INCOME_TOTAL"), 4)))
        register_feature("annuity_income_ratio", table_name, "AMT_ANNUITY, AMT_INCOME_TOTAL",
            "AMT_ANNUITY / AMT_INCOME_TOTAL", "derivation",
            "double", "Razão anuidade/renda", "NULL quando renda é NULL ou zero")
    
    # credit_goods_ratio = AMT_CREDIT / AMT_GOODS_PRICE
    if "AMT_CREDIT" in cols and "AMT_GOODS_PRICE" in cols:
        df = df.withColumn("credit_goods_ratio",
            F.when(F.col("AMT_GOODS_PRICE").isNull() | (F.col("AMT_GOODS_PRICE") == 0), None)
             .otherwise(F.round(F.col("AMT_CREDIT") / F.col("AMT_GOODS_PRICE"), 4)))
        register_feature("credit_goods_ratio", table_name, "AMT_CREDIT, AMT_GOODS_PRICE",
            "AMT_CREDIT / AMT_GOODS_PRICE", "derivation",
            "double", "Razão crédito/preço do bem", "NULL quando preço é NULL ou zero")
    
    # credit_term_ratio = AMT_ANNUITY / AMT_CREDIT (proxy para prazo do empréstimo)
    if "AMT_ANNUITY" in cols and "AMT_CREDIT" in cols:
        df = df.withColumn("credit_term_ratio",
            F.when(F.col("AMT_CREDIT").isNull() | (F.col("AMT_CREDIT") == 0), None)
             .otherwise(F.round(F.col("AMT_ANNUITY") / F.col("AMT_CREDIT"), 4)))
        register_feature("credit_term_ratio", table_name, "AMT_ANNUITY, AMT_CREDIT",
            "AMT_ANNUITY / AMT_CREDIT", "derivation",
            "double", "Proxy para prazo do empréstimo", "NULL quando crédito é NULL ou zero")
    
    # family_size_per_child = CNT_FAM_MEMBERS / (CNT_CHILDREN + 1)
    if "CNT_FAM_MEMBERS" in cols and "CNT_CHILDREN" in cols:
        df = df.withColumn("family_size_per_child",
            F.when(F.col("CNT_FAM_MEMBERS").isNull(), None)
             .otherwise(F.round(F.col("CNT_FAM_MEMBERS") / (F.col("CNT_CHILDREN") + 1), 2)))
        register_feature("family_size_per_child", table_name, "CNT_FAM_MEMBERS, CNT_CHILDREN",
            "CNT_FAM_MEMBERS / (CNT_CHILDREN + 1)", "derivation",
            "double", "Tamanho da família por filho", "NULL quando CNT_FAM_MEMBERS é NULL")
    
    return df

# Preparar features de application_train e application_test
print("=" * 70)
print("PREPARAÇÃO DAS FEATURES DA APLICAÇÃO")
print("=" * 70)

df_app_train = prepare_application_features(spark.table(SILVER_APP_TRAIN), SILVER_APP_TRAIN)
df_app_test = prepare_application_features(spark.table(SILVER_APP_TEST), SILVER_APP_TEST)

train_feature_count = len(df_app_train.columns)
test_feature_count = len(df_app_test.columns)

print(f"\n   application_train: {train_feature_count} features")
print(f"   application_test: {test_feature_count} features")
print(f"   Diferença (esperada = 0, já que TARGET é excluído): {train_feature_count - test_feature_count}")

print("\n✅ Features da aplicação preparadas!")

In [0]:
# ============================================================================
# CÉLULA 5 — Agregação Previous Application
# ============================================================================
# Agrega previous_application por SK_ID_CURR (1 linha por cliente).
# Cria features de contagem, valores financeiros e proporções.

print("=" * 70)
print("AGREGAÇÃO PREVIOUS APPLICATION")
print("=" * 70)

df_prev = spark.table(SILVER_PREV_APP)
prev_cols = SILVER_COLUMNS[SILVER_PREV_APP]

# Agregação por SK_ID_CURR
agg_prev = df_prev.groupBy("SK_ID_CURR").agg(
    # Contagens
    F.count("*").alias("prev_app_count"),
    F.countDistinct("SK_ID_PREV").alias("prev_app_contract_count"),
    F.sum(F.when(F.col("NAME_CONTRACT_STATUS") == "Approved", 1).otherwise(0)).alias("prev_app_approved_count"),
    F.sum(F.when(F.col("NAME_CONTRACT_STATUS") == "Refused", 1).otherwise(0)).alias("prev_app_refused_count"),
    F.sum(F.when(F.col("NAME_CONTRACT_STATUS") == "Canceled", 1).otherwise(0)).alias("prev_app_canceled_count"),
    F.sum(F.when(F.col("NAME_CONTRACT_STATUS") == "Unused offer", 1).otherwise(0)).alias("prev_app_unused_count"),
    
    # Valores financeiros
    F.avg("AMT_APPLICATION").alias("prev_app_avg_application"),
    F.max("AMT_APPLICATION").alias("prev_app_max_application"),
    F.avg("AMT_CREDIT").alias("prev_app_avg_credit"),
    F.max("AMT_CREDIT").alias("prev_app_max_credit"),
    F.avg("AMT_ANNUITY").alias("prev_app_avg_annuity"),
    F.avg("AMT_DOWN_PAYMENT").alias("prev_app_avg_down_payment"),
    
    # Temporal
    F.avg("DAYS_DECISION").alias("prev_app_avg_days_decision"),
    F.max("DAYS_DECISION").alias("prev_app_max_days_decision"),
    
    # Parcelamento
    F.avg("CNT_PAYMENT").alias("prev_app_avg_cnt_payment"),
)

# Proporção aprovadas (aprovadas / total)
agg_prev = agg_prev.withColumn("prev_app_approval_rate",
    F.when(F.col("prev_app_count") == 0, None)
     .otherwise(F.round(F.col("prev_app_approved_count") / F.col("prev_app_count"), 4)))

# Proporção recusadas
agg_prev = agg_prev.withColumn("prev_app_refusal_rate",
    F.when(F.col("prev_app_count") == 0, None)
     .otherwise(F.round(F.col("prev_app_refused_count") / F.col("prev_app_count"), 4)))

prev_feature_count = len(agg_prev.columns)
print(f"\n   Features criadas: {prev_feature_count - 1} (excluindo SK_ID_CURR)")
print(f"   Clientes com histórico: {agg_prev.count():,}")

# Registrar features no catálogo
for col_name in agg_prev.columns:
    if col_name != "SK_ID_CURR":
        register_feature(col_name, SILVER_PREV_APP, "SK_ID_CURR, NAME_CONTRACT_STATUS, AMT_*",
            f"Agregação de {col_name} por SK_ID_CURR", "groupBy + agg",
            "double" if col_name not in ["prev_app_count", "prev_app_contract_count",
            "prev_app_approved_count", "prev_app_refused_count",
            "prev_app_canceled_count", "prev_app_unused_count"] else "long",
            f"Contagem/estatística de aplicações anteriores",
            "NULL quando cliente não tem histórico de aplicações anteriores")

print("\n✅ Previous Application agregado!")

In [0]:
# ============================================================================
# CÉLULA 6 — Agregação Bureau
# ============================================================================
# Agrega bureau por SK_ID_CURR (1 linha por cliente).
# Cria features de contagem, crédito, dívida e atraso.

print("=" * 70)
print("AGREGAÇÃO BUREAU")
print("=" * 70)

df_bureau = spark.table(SILVER_BUREAU)
bureau_cols = SILVER_COLUMNS[SILVER_BUREAU]

agg_bureau = df_bureau.groupBy("SK_ID_CURR").agg(
    # Contagens
    F.count("*").alias("bureau_count"),
    F.countDistinct("SK_ID_BUREAU").alias("bureau_credit_count"),
    F.sum(F.when(F.col("CREDIT_ACTIVE") == "Active", 1).otherwise(0)).alias("bureau_active_count"),
    F.sum(F.when(F.col("CREDIT_ACTIVE") == "Closed", 1).otherwise(0)).alias("bureau_closed_count"),
    F.sum(F.when(F.col("CREDIT_ACTIVE").isin(["Bad debt", "Sold"]), 1).otherwise(0)).alias("bureau_bad_sold_count"),
    
    # Valores de crédito
    F.sum("AMT_CREDIT_SUM").alias("bureau_sum_credit"),
    F.avg("AMT_CREDIT_SUM").alias("bureau_avg_credit"),
    F.max("AMT_CREDIT_SUM").alias("bureau_max_credit"),
    
    # Valores de dívida
    F.sum("AMT_CREDIT_SUM_DEBT").alias("bureau_sum_debt"),
    F.avg("AMT_CREDIT_SUM_DEBT").alias("bureau_avg_debt"),
    
    # Atraso
    F.max("AMT_CREDIT_MAX_OVERDUE").alias("bureau_max_overdue"),
    F.sum("AMT_CREDIT_SUM_OVERDUE").alias("bureau_sum_overdue"),
    F.sum(F.when(F.col("CREDIT_DAY_OVERDUE") > 0, 1).otherwise(0)).alias("bureau_overdue_count"),
    
    # Temporal
    F.avg("DAYS_CREDIT").alias("bureau_avg_days_credit"),
    F.min("DAYS_CREDIT").alias("bureau_min_days_credit"),
    
    # Prolongamentos
    F.sum("CNT_CREDIT_PROLONG").alias("bureau_prolong_count"),
)

# Proporção dívida/crédito
agg_bureau = agg_bureau.withColumn("bureau_debt_to_credit_ratio",
    F.when(F.col("bureau_sum_credit").isNull() | (F.col("bureau_sum_credit") == 0), None)
     .otherwise(F.round(F.col("bureau_sum_debt") / F.col("bureau_sum_credit"), 4)))

# Proporção ativos
agg_bureau = agg_bureau.withColumn("bureau_active_rate",
    F.when(F.col("bureau_count") == 0, None)
     .otherwise(F.round(F.col("bureau_active_count") / F.col("bureau_count"), 4)))

print(f"\n   Features criadas: {len(agg_bureau.columns) - 1}")
print(f"   Clientes com histórico bureau: {agg_bureau.count():,}")

# Registrar features
for col_name in agg_bureau.columns:
    if col_name != "SK_ID_CURR":
        register_feature(col_name, SILVER_BUREAU, "SK_ID_CURR, CREDIT_ACTIVE, AMT_CREDIT_*",
            f"Agregação de {col_name} por SK_ID_CURR", "groupBy + agg",
            "double", "Estatísticas de crédito bureau",
            "NULL quando cliente não tem registros no bureau")

print("\n✅ Bureau agregado!")

In [0]:
# ============================================================================
# CÉLULA 7 — Agregação Bureau Balance
# ============================================================================
# Agrega bureau_balance primeiro por SK_ID_BUREAU e depois por SK_ID_CURR.
# NÃO faz JOIN direto entre bureau_balance e application.

print("=" * 70)
print("AGREGAÇÃO BUREAU BALANCE")
print("=" * 70)

df_bb = spark.table(SILVER_BUREAU_BALANCE)
df_bureau = spark.table(SILVER_BUREAU)

# Passo 1: Agregar bureau_balance por SK_ID_BUREAU
bb_by_bureau = df_bb.groupBy("SK_ID_BUREAU").agg(
    F.count("*").alias("bb_months_count"),
    F.max("MONTHS_BALANCE").alias("bb_latest_month"),
    F.min("MONTHS_BALANCE").alias("bb_earliest_month"),
    # STATUS 'C' = fechado, 'X' = sem DPD, '0' = pago no dia, '1'-'5' = DPD crescente
    F.sum(F.when(F.col("STATUS") == "C", 1).otherwise(0)).alias("bb_status_c_count"),
    F.sum(F.when(F.col("STATUS") == "X", 1).otherwise(0)).alias("bb_status_x_count"),
    F.sum(F.when(F.col("STATUS") == "0", 1).otherwise(0)).alias("bb_status_0_count"),
    F.sum(F.when(F.col("STATUS").isin(["1", "2", "3", "4", "5"]), 1).otherwise(0)).alias("bb_status_dpd_count"),
    F.sum(F.when(F.col("STATUS") == "5", 1).otherwise(0)).alias("bb_status_5_count"),
)

# Passo 2: JOIN com bureau para obter SK_ID_CURR, depois agregar por SK_ID_CURR
bb_with_curr = bb_by_bureau.join(
    df_bureau.select("SK_ID_BUREAU", "SK_ID_CURR"),
    "SK_ID_BUREAU", "inner"
)

agg_bb = bb_with_curr.groupBy("SK_ID_CURR").agg(
    F.sum("bb_months_count").alias("bb_total_months"),
    F.avg("bb_months_count").alias("bb_avg_months_per_credit"),
    F.max("bb_latest_month").alias("bb_latest_month"),
    F.sum("bb_status_c_count").alias("bb_closed_count"),
    F.sum("bb_status_x_count").alias("bb_no_dpd_count"),
    F.sum("bb_status_0_count").alias("bb_paid_on_time_count"),
    F.sum("bb_status_dpd_count").alias("bb_dpd_count"),
    F.sum("bb_status_5_count").alias("bb_dpd_120_plus_count"),
)

# Proporção de meses com DPD
agg_bb = agg_bb.withColumn("bb_dpd_rate",
    F.when(F.col("bb_total_months") == 0, None)
     .otherwise(F.round(F.col("bb_dpd_count") / F.col("bb_total_months"), 4)))

print(f"\n   Features criadas: {len(agg_bb.columns) - 1}")
print(f"   Clientes com histórico bureau_balance: {agg_bb.count():,}")

# Registrar features
for col_name in agg_bb.columns:
    if col_name != "SK_ID_CURR":
        register_feature(col_name, "bureau_balance + bureau",
            "SK_ID_BUREAU, MONTHS_BALANCE, STATUS, SK_ID_CURR",
            f"Agregação de {col_name} (bureau_balance → SK_ID_BUREAU → SK_ID_CURR)",
            "groupBy + agg (2 níveis)", "double",
            "Histórico mensal de status de crédito bureau",
            "NULL quando cliente não tem bureau_balance")

print("\n✅ Bureau Balance agregado!")

In [0]:
# ============================================================================
# CÉLULA 8 — Agregação POS Cash
# ============================================================================
# Agrega pos_cash_balance por SK_ID_CURR.
# Cria features de contagem, meses, parcelas e DPD.

print("=" * 70)
print("AGREGAÇÃO POS CASH")
print("=" * 70)

df_pos = spark.table(SILVER_POS_CASH)

agg_pos = df_pos.groupBy("SK_ID_CURR").agg(
    # Contagens
    F.count("*").alias("pos_cash_count"),
    F.countDistinct("SK_ID_PREV").alias("pos_cash_contract_count"),
    
    # Meses
    F.max("MONTHS_BALANCE").alias("pos_cash_latest_month"),
    F.min("MONTHS_BALANCE").alias("pos_cash_earliest_month"),
    (F.max("MONTHS_BALANCE") - F.min("MONTHS_BALANCE")).alias("pos_cash_months_span"),
    
    # Parcelas
    F.avg("CNT_INSTALMENT").alias("pos_cash_avg_instalment"),
    F.avg("CNT_INSTALMENT_FUTURE").alias("pos_cash_avg_instalment_future"),
    F.max("CNT_INSTALMENT_FUTURE").alias("pos_cash_max_instalment_future"),
    
    # Status
    F.sum(F.when(F.col("NAME_CONTRACT_STATUS") == "Active", 1).otherwise(0)).alias("pos_cash_active_count"),
    F.sum(F.when(F.col("NAME_CONTRACT_STATUS") == "Completed", 1).otherwise(0)).alias("pos_cash_completed_count"),
    
    # DPD
    F.max("SK_DPD").alias("pos_cash_max_dpd"),
    F.max("SK_DPD_DEF").alias("pos_cash_max_dpd_def"),
    F.sum(F.when(F.col("SK_DPD") > 0, 1).otherwise(0)).alias("pos_cash_dpd_count"),
    F.avg("SK_DPD").alias("pos_cash_avg_dpd"),
)

# Proporção ativos
agg_pos = agg_pos.withColumn("pos_cash_active_rate",
    F.when(F.col("pos_cash_count") == 0, None)
     .otherwise(F.round(F.col("pos_cash_active_count") / F.col("pos_cash_count"), 4)))

print(f"\n   Features criadas: {len(agg_pos.columns) - 1}")
print(f"   Clientes com histórico POS Cash: {agg_pos.count():,}")

for col_name in agg_pos.columns:
    if col_name != "SK_ID_CURR":
        register_feature(col_name, SILVER_POS_CASH, "SK_ID_CURR, MONTHS_BALANCE, SK_DPD",
            f"Agregação de {col_name} por SK_ID_CURR", "groupBy + agg",
            "double", "Histórico de saldo POS Cash",
            "NULL quando cliente não tem histórico POS Cash")

print("\n✅ POS Cash agregado!")

In [0]:
# ============================================================================
# CÉLULA 9 — Agregação Credit Card
# ============================================================================
# Agrega credit_card_balance por SK_ID_CURR.
# Cria features de saldo, limite, utilização e DPD.
# Valores negativos preservados da Silver não são removidos aqui.

print("=" * 70)
print("AGREGAÇÃO CREDIT CARD")
print("=" * 70)

df_cc = spark.table(SILVER_CC_BALANCE)
cc_cols = SILVER_COLUMNS[SILVER_CC_BALANCE]

agg_cc = df_cc.groupBy("SK_ID_CURR").agg(
    # Contagens
    F.count("*").alias("cc_count"),
    F.countDistinct("SK_ID_PREV").alias("cc_contract_count"),
    
    # Meses
    F.max("MONTHS_BALANCE").alias("cc_latest_month"),
    (F.max("MONTHS_BALANCE") - F.min("MONTHS_BALANCE")).alias("cc_months_span"),
    
    # Saldo
    F.avg("AMT_BALANCE").alias("cc_avg_balance"),
    F.max("AMT_BALANCE").alias("cc_max_balance"),
    
    # Limite
    F.avg("AMT_CREDIT_LIMIT_ACTUAL").alias("cc_avg_credit_limit"),
    F.max("AMT_CREDIT_LIMIT_ACTUAL").alias("cc_max_credit_limit"),
    
    # Saques
    F.avg("AMT_DRAWINGS_CURRENT").alias("cc_avg_drawings"),
    F.max("AMT_DRAWINGS_CURRENT").alias("cc_max_drawings"),
    
    # Pagamentos
    F.avg("AMT_PAYMENT_CURRENT").alias("cc_avg_payment"),
    F.avg("AMT_PAYMENT_TOTAL_CURRENT").alias("cc_avg_total_payment"),
    
    # Recebíveis
    F.avg("AMT_TOTAL_RECEIVABLE").alias("cc_avg_receivable"),
    
    # Contagem de saques
    F.avg("CNT_DRAWINGS_CURRENT").alias("cc_avg_drawings_count"),
    
    # DPD
    F.max("SK_DPD").alias("cc_max_dpd"),
    F.max("SK_DPD_DEF").alias("cc_max_dpd_def"),
    F.sum(F.when(F.col("SK_DPD") > 0, 1).otherwise(0)).alias("cc_dpd_count"),
)

# Utilização média = AVG(AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL)
df_cc_util = df_cc.withColumn("utilization",
    F.when(F.col("AMT_CREDIT_LIMIT_ACTUAL").isNull() | (F.col("AMT_CREDIT_LIMIT_ACTUAL") == 0), None)
     .otherwise(F.col("AMT_BALANCE") / F.col("AMT_CREDIT_LIMIT_ACTUAL"))
)

agg_util = df_cc_util.groupBy("SK_ID_CURR").agg(
    F.avg("utilization").alias("cc_avg_utilization"),
    F.max("utilization").alias("cc_max_utilization"),
)

agg_cc = agg_cc.join(agg_util, "SK_ID_CURR", "left")

# Contagem de saldos negativos (preservados da Silver)
if "FLAG_AMT_BALANCE_NEGATIVE" in cc_cols:
    neg_count = df_cc.groupBy("SK_ID_CURR").agg(
        F.sum(F.when(F.col("FLAG_AMT_BALANCE_NEGATIVE") == 1, 1).otherwise(0)).alias("cc_negative_balance_count")
    )
    agg_cc = agg_cc.join(neg_count, "SK_ID_CURR", "left")

# Proporção meses com DPD
agg_cc = agg_cc.withColumn("cc_dpd_rate",
    F.when(F.col("cc_count") == 0, None)
     .otherwise(F.round(F.col("cc_dpd_count") / F.col("cc_count"), 4)))

print(f"\n   Features criadas: {len(agg_cc.columns) - 1}")
print(f"   Clientes com histórico credit card: {agg_cc.count():,}")

for col_name in agg_cc.columns:
    if col_name != "SK_ID_CURR":
        register_feature(col_name, SILVER_CC_BALANCE, "SK_ID_CURR, AMT_BALANCE, AMT_CREDIT_LIMIT_ACTUAL",
            f"Agregação de {col_name} por SK_ID_CURR", "groupBy + agg",
            "double", "Histórico de saldo de cartão de crédito",
            "NULL quando cliente não tem histórico de cartão")

print("\n✅ Credit Card agregado!")

In [0]:
# ============================================================================
# CÉLULA 10 — Agregação Installments Payments
# ============================================================================
# Agrega installments_payments por SK_ID_CURR.
# NÃO deduplica os 653K pagamentos parciais legítimos.
# Cria features de atraso, pagamento e razão pago/previsto.

print("=" * 70)
print("AGREGAÇÃO INSTALLMENTS PAYMENTS")
print("=" * 70)

df_inst = spark.table(SILVER_INST_PAY)
inst_cols = SILVER_COLUMNS[SILVER_INST_PAY]

# Feature de atraso no nível de linha
df_inst_enriched = df_inst.withColumn("payment_delay_days",
    F.when(F.col("DAYS_ENTRY_PAYMENT").isNull() | F.col("DAYS_INSTALMENT").isNull(), None)
     .otherwise(F.col("DAYS_ENTRY_PAYMENT") - F.col("DAYS_INSTALMENT"))
).withColumn("payment_ratio",
    F.when(F.col("AMT_INSTALMENT").isNull() | (F.col("AMT_INSTALMENT") == 0), None)
     .otherwise(F.col("AMT_PAYMENT") / F.col("AMT_INSTALMENT"))
)

agg_inst = df_inst_enriched.groupBy("SK_ID_CURR").agg(
    # Contagens
    F.count("*").alias("inst_count"),
    F.countDistinct("SK_ID_PREV").alias("inst_contract_count"),
    
    # Valores pagos e previstos
    F.sum("AMT_PAYMENT").alias("inst_total_paid"),
    F.sum("AMT_INSTALMENT").alias("inst_total_instalment"),
    F.avg("AMT_PAYMENT").alias("inst_avg_payment"),
    F.avg("AMT_INSTALMENT").alias("inst_avg_instalment"),
    F.max("AMT_INSTALMENT").alias("inst_max_instalment"),
    F.max("AMT_PAYMENT").alias("inst_max_payment"),
    
    # Atraso
    F.avg("payment_delay_days").alias("inst_avg_delay_days"),
    F.max("payment_delay_days").alias("inst_max_delay_days"),
    F.sum(F.when(F.col("payment_delay_days") > 0, 1).otherwise(0)).alias("inst_late_count"),
    F.sum(F.when(F.col("payment_delay_days") < 0, 1).otherwise(0)).alias("inst_early_count"),
    
    # Sem pagamento
    F.sum(F.when(F.col("DAYS_ENTRY_PAYMENT").isNull(), 1).otherwise(0)).alias("inst_missing_payment_count"),
    
    # Razão pago/previsto
    F.avg("payment_ratio").alias("inst_avg_payment_ratio"),
)

# Proporção de pagamentos atrasados
agg_inst = agg_inst.withColumn("inst_late_rate",
    F.when(F.col("inst_count") == 0, None)
     .otherwise(F.round(F.col("inst_late_count") / F.col("inst_count"), 4)))

# Proporção de pagamentos ausentes
agg_inst = agg_inst.withColumn("inst_missing_rate",
    F.when(F.col("inst_count") == 0, None)
     .otherwise(F.round(F.col("inst_missing_payment_count") / F.col("inst_count"), 4)))

# Razão total pago / total previsto
agg_inst = agg_inst.withColumn("inst_total_paid_to_instalment_ratio",
    F.when(F.col("inst_total_instalment").isNull() | (F.col("inst_total_instalment") == 0), None)
     .otherwise(F.round(F.col("inst_total_paid") / F.col("inst_total_instalment"), 4)))

print(f"\n   Features criadas: {len(agg_inst.columns) - 1}")
print(f"   Clientes com histórico installments: {agg_inst.count():,}")

for col_name in agg_inst.columns:
    if col_name != "SK_ID_CURR":
        register_feature(col_name, SILVER_INST_PAY,
            "SK_ID_CURR, DAYS_INSTALMENT, DAYS_ENTRY_PAYMENT, AMT_INSTALMENT, AMT_PAYMENT",
            f"Agregação de {col_name} por SK_ID_CURR", "groupBy + agg",
            "double", "Histórico de pagamentos de parcelas",
            "NULL quando cliente não tem histórico de parcelas")

print("\n✅ Installments Payments agregado!")

In [0]:
# ============================================================================
# CÉLULA 11 — Consolidação das Features
# ============================================================================
# Faz LEFT JOIN da aplicação com todas as agregações históricas.
# Cada agregação já tem 1 linha por SK_ID_CURR → sem row explosion.

print("=" * 70)
print("CONSOLIDAÇÃO DAS FEATURES")
print("=" * 70)

def consolidate_features(df_app):
    """Faz LEFT JOIN da aplicação com todas as agregações históricas."""
    df = df_app
    
    # LEFT JOIN com cada agregação (usando syntax usingColumn para evitar duplicação de chave)
    aggregations = [agg_prev, agg_bureau, agg_bb, agg_pos, agg_cc, agg_inst]
    
    for agg_df in aggregations:
        df = df.join(agg_df, "SK_ID_CURR", "left")
    
    return df

# Consolidar features para train e test
df_gold_train = consolidate_features(df_app_train)
df_gold_test = consolidate_features(df_app_test)

train_total_cols = len(df_gold_train.columns)
test_total_cols = len(df_gold_test.columns)

print(f"\n   Gold Train: {train_total_cols} colunas")
print(f"   Gold Test: {test_total_cols} colunas")
print(f"   Diferença (esperada = 0): {train_total_cols - test_total_cols}")

print("\n✅ Features consolidadas!")

In [0]:
# ============================================================================
# CÉLULA 12 — Separação Train/Test
# ============================================================================
# Adiciona TARGET ao train (apenas como variável alvo, não como feature).
# Garante que train e test tenham o mesmo conjunto de features.

print("=" * 70)
print("SEPARAÇÃO TRAIN/TEST")
print("=" * 70)

# Adicionar TARGET ao train (do application_train original)
df_silver_train = spark.table(SILVER_APP_TRAIN)
df_gold_train = df_gold_train.join(
    df_silver_train.select("SK_ID_CURR", "TARGET"),
    "SK_ID_CURR", "left"
)

# Garantir que test NÃO tem TARGET
if "TARGET" in df_gold_test.columns:
    df_gold_test = df_gold_test.drop("TARGET")

# Verificar alinhamento de schema (excluindo TARGET)
train_cols_no_target = sorted([c for c in df_gold_train.columns if c != "TARGET"])
test_cols = sorted(df_gold_test.columns)

print(f"\n   Train: {len(df_gold_train.columns)} colunas (incluindo TARGET)")
print(f"   Test: {len(df_gold_test.columns)} colunas")

# Features em train mas não em test (excluindo TARGET)
train_only = set(train_cols_no_target) - set(test_cols)
test_only = set(test_cols) - set(train_cols_no_target)

if train_only:
    print(f"\n   ⚠️ Features no Train mas não no Test: {train_only}")
else:
    print(f"\n   ✅ Nenhuma feature exclusiva do Train")

if test_only:
    print(f"   ⚠️ Features no Test mas não no Train: {test_only}")
else:
    print(f"   ✅ Nenhuma feature exclusiva do Test")

# Validação de vazamento: TARGET não pode estar entre as features
feature_names = [c for c in df_gold_train.columns if c != "TARGET" and c != "SK_ID_CURR"]
if "TARGET" in feature_names:
    print("\n   ❌ ERRO CRÍTICO: TARGET encontrada entre as features!")
else:
    print(f"\n   ✅ Controle de vazamento: TARGET não está entre as features")

print(f"\n   Total de features: {len(feature_names)}")
print(f"   Train: {df_gold_train.count():,} registros")
print(f"   Test: {df_gold_test.count():,} registros")

print("\n✅ Separação Train/Test concluída!")

In [0]:
# ============================================================================
# CÉLULA 13 — Validações de Cardinalidade e Integridade
# ============================================================================
# Verifica duplicidades de SK_ID_CURR, valores infinitos e impossíveis.

sep = "─" * 60
print("=" * 70)
print("VALIDAÇÕES DE CARDINALIDADE E INTEGRIDADE")
print("=" * 70)

for label, df_gold in [("Train", df_gold_train), ("Test", df_gold_test)]:
    print(f"\n{'─' * 60}")
    print(f"📊 {label}")
    print(f"{'─' * 60}")
    
    rc = df_gold.count()
    distinct_curr = df_gold.select("SK_ID_CURR").distinct().count()
    dups = rc - distinct_curr
    null_curr = df_gold.filter(F.col("SK_ID_CURR").isNull()).count()
    
    print(f"   Registros: {rc:,}")
    print(f"   SK_ID_CURR distintos: {distinct_curr:,}")
    print(f"   Duplicidades SK_ID_CURR: {dups}")
    print(f"   SK_ID_CURR NULL: {null_curr}")
    
    if dups == 0:
        print(f"   ✅ 0 duplicidades de SK_ID_CURR")
    else:
        print(f"   ❌ {dups} duplicidades encontradas!")
    
    if null_curr == 0:
        print(f"   ✅ 0 SK_ID_CURR NULL")
    else:
        print(f"   ❌ {null_curr} SK_ID_CURR NULL!")

# Comparação com Silver
print(f"\n{'─' * 60}")
print("COMPARAÇÃO SILVER → GOLD")
print(f"{'─' * 60}")

silver_train_count = spark.table(SILVER_APP_TRAIN).count()
silver_test_count = spark.table(SILVER_APP_TEST).count()
gold_train_count = df_gold_train.count()
gold_test_count = df_gold_test.count()

print(f"   Silver Train: {silver_train_count:,} → Gold Train: {gold_train_count:,} (delta: {gold_train_count - silver_train_count})")
print(f"   Silver Test: {silver_test_count:,} → Gold Test: {gold_test_count:,} (delta: {gold_test_count - silver_test_count})")

if gold_train_count == silver_train_count:
    print(f"   ✅ Train: nenhum cliente perdido")
else:
    print(f"   ⚠️ Train: {abs(gold_train_count - silver_train_count)} clientes perdidos")

if gold_test_count == silver_test_count:
    print(f"   ✅ Test: nenhum cliente perdido")
else:
    print(f"   ⚠️ Test: {abs(gold_test_count - silver_test_count)} clientes perdidos")

print("\n✅ Validações de cardinalidade concluídas!")

In [0]:
# ============================================================================
# CÉLULA 14 — Validação de Schema Train × Test
# ============================================================================
# Verifica que train e test possuem exatamente o mesmo conjunto de features.

sep = "─" * 60
print("=" * 70)
print("VALIDAÇÃO DE SCHEMA TRAIN × TEST")
print("=" * 70)

train_cols = sorted([c for c in df_gold_train.columns if c != "TARGET"])
test_cols = sorted(df_gold_test.columns)

print(f"\n   Train (sem TARGET): {len(train_cols)} colunas")
print(f"   Test: {len(test_cols)} colunas")

cols_match = train_cols == test_cols
if cols_match:
    print(f"   ✅ Schema alinhado: Train e Test têm as mesmas features")
else:
    train_set = set(train_cols)
    test_set = set(test_cols)
    only_train = train_set - test_set
    only_test = test_set - train_set
    if only_train:
        print(f"   ⚠️ Apenas no Train: {only_train}")
    if only_test:
        print(f"   ⚠️ Apenas no Test: {only_test}")

# Comparação de tipos
print(f"\n{'─' * 60}")
print("COMPARAÇÃO DE TIPOS")
print(f"{'─' * 60}")

train_types = {f.name: f.dataType.simpleString() for f in df_gold_train.schema.fields if f.name != "TARGET"}
test_types = {f.name: f.dataType.simpleString() for f in df_gold_test.schema.fields}

type_mismatches = []
for col_name in train_cols:
    if col_name in test_types:
        if train_types[col_name] != test_types[col_name]:
            type_mismatches.append((col_name, train_types[col_name], test_types[col_name]))

if type_mismatches:
    print(f"   ⚠️ {len(type_mismatches)} colunas com tipo divergente:")
    for cn, tt, et in type_mismatches[:10]:
        print(f"      {cn}: train={tt}, test={et}")
else:
    print(f"   ✅ Todos os tipos alinhados entre Train e Test")

# Verificar explicitamente que TARGET está apenas no Train
print(f"\n{'─' * 60}")
print("CONTROLE DE VAZAMENTO")
print(f"{'─' * 60}")

has_target_train = "TARGET" in df_gold_train.columns
has_target_test = "TARGET" in df_gold_test.columns

print(f"   TARGET no Train: {'✅ Sim' if has_target_train else '❌ Não'}")
print(f"   TARGET no Test: {'❌ Sim (ERRO!)' if has_target_test else '✅ Não'}")

if has_target_train and not has_target_test:
    print(f"   ✅ Controle de vazamento: TARGET apenas no Train")
else:
    print(f"   ❌ ERRO: Configuração incorreta de TARGET")

print("\n✅ Validação de schema concluída!")

In [0]:
# ============================================================================
# CÉLULA 15 — Perfil de NULL e Estatísticas
# ============================================================================
# Calcula NULLs por feature e estatísticas das colunas numéricas no Gold Train.

sep = "─" * 60
print("=" * 70)
print("PERFIL DE NULL E ESTATÍSTICAS")
print("=" * 70)

train_rc = df_gold_train.count()

# NULLs por coluna (top 20 com mais NULLs)
print(f"\n{'─' * 60}")
print("TOP 20 FEATURES COM MAIS NULLs (Train)")
print(f"{'─' * 60}")

null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) 
    for c in df_gold_train.columns if c != "TARGET"]
null_row = df_gold_train.agg(*null_exprs).collect()[0]

null_info = []
for c in df_gold_train.columns:
    if c == "TARGET":
        continue
    n = null_row[c] if null_row[c] else 0
    pct = n / train_rc * 100
    null_info.append((c, n, pct))

null_info.sort(key=lambda x: x[1], reverse=True)

print(f"   {'Feature':<40} {'NULLs':>10} {'%':>8}")
print(f"   {'─' * 60}")
for c, n, pct in null_info[:20]:
    print(f"   {c:<40} {n:>10,} {pct:>7.2f}%")

# Features sem NULLs
no_null = [c for c, n, _ in null_info if n == 0]
null_features = len(null_info) - len(no_null)
print(f"\n   Features sem NULLs: {len(no_null)}")
print(f"   Features com NULLs: {null_features}")

# Estatísticas das features numéricas derivadas
print(f"\n{'─' * 60}")
print("ESTATÍSTICAS DAS FEATURES DERIVADAS")
print(f"{'─' * 60}")

derived_numeric = ["age_years", "employed_years", "credit_income_ratio", 
    "annuity_income_ratio", "credit_goods_ratio", "credit_term_ratio",
    "family_size_per_child", "prev_app_count", "bureau_count",
    "pos_cash_count", "cc_count", "inst_count",
    "inst_avg_delay_days", "inst_max_delay_days", "inst_avg_payment_ratio",
    "cc_avg_utilization", "bureau_debt_to_credit_ratio", "prev_app_approval_rate"]

available_derived = [c for c in derived_numeric if c in df_gold_train.columns]

if available_derived:
    stats_df = df_gold_train.select(available_derived).summary("min", "max", "mean", "50%", "stddev")
    display(stats_df)
else:
    print("   Nenhuma feature derivada encontrada")

print("\n✅ Perfil de NULL e estatísticas concluído!")

In [0]:
# ============================================================================
# CÉLULA 16 — Criação/Atualização do Feature Catalog
# ============================================================================
# Cria credit_risk.gold.feature_catalog documentando cada feature criada.

print("=" * 70)
print("FEATURE CATALOG")
print("=" * 70)

print(f"\n   Features registradas no catálogo: {len(FEATURE_CATALOG)}")

# Schema do feature catalog
catalog_schema = StructType([
    StructField("feature_name", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("source_columns", StringType(), True),
    StructField("feature_definition", StringType(), True),
    StructField("aggregation_method", StringType(), True),
    StructField("data_type", StringType(), True),
    StructField("business_meaning", StringType(), True),
    StructField("null_semantics", StringType(), True),
    StructField("leakage_check", StringType(), True),
    StructField("ml_feature_flag", BooleanType(), True),
    StructField("created_at", TimestampType(), True),
])

catalog_df = spark.createDataFrame(FEATURE_CATALOG, schema=catalog_schema)

print(f"   Gravando em {GOLD_FEATURE_CATALOG}...")
catalog_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(GOLD_FEATURE_CATALOG)

print(f"   ✅ {len(FEATURE_CATALOG)} features documentadas em {GOLD_FEATURE_CATALOG}")

# Mostrar amostra do catálogo
print(f"\n   Amostra (primeiras 10 features):")
display(spark.table(GOLD_FEATURE_CATALOG).select(
    "feature_name", "source_table", "aggregation_method", "leakage_check"
).limit(10))

print("\n✅ Feature Catalog criado!")

In [0]:
# ============================================================================
# CÉLULA 17 — Persistência das Tabelas Gold em Delta
# ============================================================================
# Grava as tabelas Gold em Delta Lake.

print("=" * 70)
print("PERSISTÊNCIA DAS TABELAS GOLD")
print("=" * 70)

write_start = datetime.now(timezone.utc)

# Gravar Train
print(f"\n   Gravando {GOLD_TRAIN_TABLE}...")
df_gold_train.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(GOLD_TRAIN_TABLE)
print(f"   ✅ Train gravado: {df_gold_train.count():,} registros, {len(df_gold_train.columns)} colunas")

# Gravar Test
print(f"\n   Gravando {GOLD_TEST_TABLE}...")
df_gold_test.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(GOLD_TEST_TABLE)
print(f"   ✅ Test gravado: {df_gold_test.count():,} registros, {len(df_gold_test.columns)} colunas")

write_end = datetime.now(timezone.utc)
write_duration = (write_end - write_start).total_seconds()

EXEC_END = datetime.now(timezone.utc)
TOTAL_DURATION = (EXEC_END - EXEC_START).total_seconds()

print(f"\n   Tempo de escrita: {write_duration:.1f}s")
print(f"   Tempo total: {TOTAL_DURATION:.1f}s")

print("\n✅ Tabelas Gold persistidas!")

In [0]:
# ============================================================================
# CÉLULA 18 — Auditoria da Execução
# ============================================================================
# Registra a execução em credit_risk.gold.audit_features (APPEND).

print("=" * 70)
print("AUDITORIA DA EXECUÇÃO")
print("=" * 70)

# Calcular métricas para auditoria
train_count = df_gold_train.count()
test_count = df_gold_test.count()
train_distinct = df_gold_train.select("SK_ID_CURR").distinct().count()
test_distinct = df_gold_test.select("SK_ID_CURR").distinct().count()
train_dups = train_count - train_distinct
test_dups = test_count - test_distinct

feature_count = len([c for c in df_gold_train.columns if c != "TARGET" and c != "SK_ID_CURR"])

# Features com NULLs
null_feature_count = sum(1 for c, n, _ in null_info if n > 0)

audit_record = {
    "execution_timestamp": EXECUTION_TIMESTAMP,
    "execution_id": EXECUTION_ID,
    "notebook_name": NOTEBOOK_NAME,
    "status": "SUCCESS",
    "train_row_count": train_count,
    "test_row_count": test_count,
    "train_distinct_customers": train_distinct,
    "test_distinct_customers": test_distinct,
    "train_duplicate_customers": train_dups,
    "test_duplicate_customers": test_dups,
    "feature_count": feature_count,
    "null_feature_count": null_feature_count,
    "execution_duration_seconds": float(TOTAL_DURATION),
    "source_tables": ", ".join(ALL_SILVER_TABLES),
    "target_tables": f"{GOLD_TRAIN_TABLE}, {GOLD_TEST_TABLE}, {GOLD_FEATURE_CATALOG}",
    "error_message": "",
}

audit_schema = StructType([
    StructField("execution_timestamp", TimestampType(), True),
    StructField("execution_id", StringType(), True),
    StructField("notebook_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("train_row_count", LongType(), True),
    StructField("test_row_count", LongType(), True),
    StructField("train_distinct_customers", LongType(), True),
    StructField("test_distinct_customers", LongType(), True),
    StructField("train_duplicate_customers", LongType(), True),
    StructField("test_duplicate_customers", LongType(), True),
    StructField("feature_count", IntegerType(), True),
    StructField("null_feature_count", IntegerType(), True),
    StructField("execution_duration_seconds", DoubleType(), True),
    StructField("source_tables", StringType(), True),
    StructField("target_tables", StringType(), True),
    StructField("error_message", StringType(), True),
])

audit_df = spark.createDataFrame([audit_record], schema=audit_schema)

print(f"\n   Gravando em {GOLD_AUDIT_TABLE}...")
audit_df.write.mode("append").format("delta").saveAsTable(GOLD_AUDIT_TABLE)
print(f"   ✅ Auditoria registrada: 1 registro em {GOLD_AUDIT_TABLE}")

print("\n✅ Auditoria concluída!")

In [0]:
# ============================================================================
# CÉLULA 19 — Resumo Final da Execução
# ============================================================================
# Exibe o resumo objetivo da execução do notebook.

sep = "=" * 50
print(sep)
print("GOLD FEATURES - RESUMO FINAL")
print(sep)

print(f"\nTabelas criadas:")
print(f"  • {GOLD_TRAIN_TABLE}")
print(f"  • {GOLD_TEST_TABLE}")
print(f"  • {GOLD_FEATURE_CATALOG}")
print(f"  • {GOLD_AUDIT_TABLE}")

print(f"\nClientes Train: {train_count:,}")
print(f"Clientes Test: {test_count:,}")
print(f"Total de features: {feature_count}")
print(f"Duplicidades Train: {train_dups}")
print(f"Duplicidades Test: {test_dups}")

# Percentual de NULLs
total_cells = train_count * feature_count
total_nulls = sum(n for _, n, _ in null_info)
null_pct = total_nulls / total_cells * 100 if total_cells > 0 else 0
print(f"Percentual de NULL: {null_pct:.2f}%")
print(f"Features com NULL: {null_feature_count}")

# Validações executadas
print(f"\nValidações executadas:")
print(f"  • Cardinalidade (SK_ID_CURR único): {'✅ PASS' if train_dups == 0 and test_dups == 0 else '❌ FAIL'}")
print(f"  • Comparação Silver → Gold: {'✅ PASS' if train_count == silver_train_count and test_count == silver_test_count else '⚠️ WARNING'}")
print(f"  • Schema Train × Test: {'✅ PASS' if cols_match else '⚠️ WARNING'}")
print(f"  • Controle de vazamento (TARGET): {'✅ PASS' if has_target_train and not has_target_test else '❌ FAIL'}")
print(f"  • Tipos alinhados: {'✅ PASS' if not type_mismatches else '⚠️ WARNING'}")

# Warnings
warnings = []
if train_dups > 0:
    warnings.append(f"{train_dups} duplicidades de SK_ID_CURR no Train")
if test_dups > 0:
    warnings.append(f"{test_dups} duplicidades de SK_ID_CURR no Test")
if train_count != silver_train_count:
    warnings.append(f"{abs(train_count - silver_train_count)} clientes perdidos no Train")
if test_count != silver_test_count:
    warnings.append(f"{abs(test_count - silver_test_count)} clientes perdidos no Test")
if not cols_match:
    warnings.append("Schema não alinhado entre Train e Test")
if null_pct > 30:
    warnings.append(f"Alto percentual de NULLs ({null_pct:.1f}%)")

# Status final
if not warnings:
    status = "SUCCESS"
else:
    status = "WARNING"

print(f"\nStatus final: {status}")

if warnings:
    print(f"\nPrincipais warnings:")
    for w in warnings:
        print(f"  ⚠️ {w}")
else:
    print(f"\n✅ Nenhum warning significativo.")

print(f"\nTempo total: {TOTAL_DURATION:.1f}s")
print(f"Execution ID: {EXECUTION_ID}")
print(f"Pipeline: {PIPELINE_VERSION}")

print(f"\n{sep}")
print(f"✅ GOLD FEATURES CONCLUÍDO: {status}")
print(sep)